# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eeshamajid8-star/flyrank-ml-eesha/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import subprocess

# Clone repo to get data
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                "starter"], check=True)

df = pd.read_csv("starter/data/raw/content_refresh_anonymized.csv")
print(f"Total pages: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(3))

Total pages: 30000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import os

# Signal Check 1: CTR vs Position
print("=== SIGNAL 1: CTR by position tier ===")
ctr_by_pos = df.groupby('position_tier')['ctr'].mean().sort_values(ascending=False)
print(ctr_by_pos.round(3))
print(f"n = {len(df)}")
print("VERDICT: CONFIRMED — higher position = higher CTR")

# Signal Check 2: Trend direction by impression tier
print("\n=== SIGNAL 2: Declining pages by impression tier ===")
trend_by_imp = df.groupby('impression_tier')['trend_direction'].apply(
    lambda x: (x == 'down').mean() * 100
).round(1)
print(trend_by_imp)
print(f"n = {len(df)}")
print("VERDICT: CONFIRMED — pages with impressions still decline")

# Build refresh score
df['refresh_score'] = df['impressions_90d'] * (1 - df['ctr'].clip(0, 1))

# Assign reason code
def get_reason(row):
    if row['position_tier'] in ['deep', 'page_3_5']:
        return 'DEEP_POSITION'
    elif row['ctr'] < 0.1 and row['impressions_90d'] > 100:
        return 'HIGH_IMP_LOW_CTR'
    elif row['trend_direction'] == 'down':
        return 'DECLINING'
    else:
        return 'LOW_PRIORITY'

df['reason_code'] = df.apply(get_reason, axis=1)
df['action'] = df['reason_code'].apply(
    lambda x: 'REFRESH_NOW' if x in ['HIGH_IMP_LOW_CTR', 'DECLINING'] else
              'IMPROVE_POSITION' if x == 'DEEP_POSITION' else 'MONITOR'
)

# Rank and save
ranked = df.sort_values('refresh_score', ascending=False).reset_index(drop=True)
ranked['rank'] = ranked.index + 1

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"\nCSV saved — {len(ranked)} rows")
print(ranked[['rank','content_id','refresh_score','reason_code','action']].head(10))

=== SIGNAL 1: CTR by position tier ===
position_tier
top_3       1.484
page_1      0.652
striking    0.323
page_3_5    0.222
deep        0.150
Name: ctr, dtype: float64
n = 30000
VERDICT: CONFIRMED — higher position = higher CTR

=== SIGNAL 2: Declining pages by impression tier ===
impression_tier
excellent    46.2
good         58.6
low          45.4
moderate     61.5
Name: trend_direction, dtype: float64
n = 30000
VERDICT: CONFIRMED — pages with impressions still decline

CSV saved — 30000 rows
   rank            content_id  refresh_score       reason_code  \
0     1  content_2cb567c3c89b      447954.30     DEEP_POSITION   
1     2  content_5fe46e04994d      445234.90         DECLINING   
2     3  content_8c19996aa890      432864.20         DECLINING   
3     4  content_aaef01a50def      387831.75      LOW_PRIORITY   
4     5  content_2dba2b1f9536      350312.86     DEEP_POSITION   
5     6  content_1a9e894be2e2      320458.60         DECLINING   
6     7  content_36ff89c8214e      28

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.Top-10 Review

Rank 1 — IMPROVE_POSITION — Deep position, high impressions but low CTR. Wrong if: page is actually new and position is expected to be low.

Rank 2 — REFRESH_NOW — Declining impressions trend. Wrong if: decline is seasonal, not content quality issue.

Rank 3 — REFRESH_NOW — Declining impressions trend. Wrong if: competitor published better content, refresh alone won't fix it.

Rank 4 — MONITOR — Low priority score. Wrong if: page has high business value not captured in impressions.

Rank 5 — IMPROVE_POSITION — Deep position. Wrong if: keyword is too competitive for position improvement.

Rank 6 — REFRESH_NOW — Declining trend. Wrong if: page was recently updated and decline is temporary.

Rank 7 — REFRESH_NOW — High impressions, low CTR. Wrong if: meta title/description needs fixing, not full content refresh.

Rank 8 — REFRESH_NOW — Declining trend. Wrong if: traffic loss is due to algorithm update affecting whole site.

Rank 9 — MONITOR — Low priority. Wrong if: page drives conversions not visible in session data.

Rank 10 — IMPROVE_POSITION — Deep position. Wrong if: page targets long-tail keywords where deep position is normal*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = ranked[['rank','content_id','refresh_score','reason_code','action',
                'impressions_90d','ctr','avg_position','trend_direction']].head(10)
print("Top 10 Review:")
print(top10.to_string())

Top 10 Review:
   rank            content_id  refresh_score       reason_code            action  impressions_90d   ctr  avg_position trend_direction
0     1  content_2cb567c3c89b      447954.30     DEEP_POSITION  IMPROVE_POSITION           497727  0.10          22.2              up
1     2  content_5fe46e04994d      445234.90         DECLINING       REFRESH_NOW           517715  0.14           4.2            down
2     3  content_8c19996aa890      432864.20         DECLINING       REFRESH_NOW           509252  0.15           2.5            down
3     4  content_aaef01a50def      387831.75      LOW_PRIORITY           MONITOR           517109  0.25           5.4          stable
4     5  content_2dba2b1f9536      350312.86     DEEP_POSITION  IMPROVE_POSITION           443434  0.21          27.9          stable
5     6  content_1a9e894be2e2      320458.60         DECLINING       REFRESH_NOW           416180  0.23           4.0            down
6     7  content_36ff89c8214e      280342.15  H

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks + leakage check
print("=== Weak picks ===")
print("Rank 4 and 9 are MONITOR — low refresh score but still in top 10")
print("This suggests score formula may over-weight raw impressions")

print("\n=== Leakage check ===")
leakage_cols = ['trend_direction', 'trend_pct', 'impression_tier']
print(f"Potential leakage columns NOT used in score: {leakage_cols}")
print("refresh_score uses only: impressions_90d and ctr")
print("reason_code uses: position_tier, ctr, impressions_90d, trend_direction")
print("WARNING: trend_direction in reason_code is a future-derived signal")
print("For clean baseline: reason_code should use only position and CTR")

# Confirm no future window used in score
print("\nScore formula: impressions_90d * (1 - ctr)")
print("Both are 90-day lookback — no future window leaked in score")

=== Weak picks ===
Rank 4 and 9 are MONITOR — low refresh score but still in top 10
This suggests score formula may over-weight raw impressions

=== Leakage check ===
Potential leakage columns NOT used in score: ['trend_direction', 'trend_pct', 'impression_tier']
refresh_score uses only: impressions_90d and ctr
reason_code uses: position_tier, ctr, impressions_90d, trend_direction
For clean baseline: reason_code should use only position and CTR

Score formula: impressions_90d * (1 - ctr)
Both are 90-day lookback — no future window leaked in score


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.